# casmi encoder training (GPU)
Generated by `scripts/kaggle_encoder_kernel.py`.

In [ ]:
# ---- settings (generated by scripts/kaggle_encoder_kernel.py at 5ea6bafed28c) ----
from pathlib import Path
OWNER = 'ograhulrajkumar'
ARMS = ['bce', 'focal_con']                  # casmi.encoder.ARMS keys, one GPU each when possible
HOURS = 6.0                # training wall budget per arm (parallel case)
SETS = {}       # Config overrides for every arm (--set key=value)
EVAL_MAX_MOL = 0  # 0 = full NP-TIMS / NP-OTHER evaluation packs
CODE_DATASET = 'casmi-encoder-code'
BUNDLE_DATASET = 'casmi-encoder-bundle'
NP_DATASET = 'casmi-db-np'
RDKIT_WHEEL_DATASET = 'metric/rdkit-2026-3-3-wheel'
WORK = Path("/kaggle/working")
OUT = WORK / "encoder"

In [ ]:
import os, sys, time, json, glob, shutil, hashlib, subprocess, traceback
from pathlib import Path

INPUT = Path("/kaggle/input")


def find_dataset(slug, owner=None):
    """Mount directory of an attached dataset, found by slug (any Kaggle mount layout)."""
    for cand in ([INPUT / "datasets" / owner / slug] if owner else []) + [INPUT / slug]:
        if cand.is_dir():
            return cand
    hits = []
    for dirpath, dirnames, _ in os.walk(INPUT):
        depth = len(Path(dirpath).relative_to(INPUT).parts)
        if depth >= 4:
            dirnames[:] = []
            continue
        for d in sorted(dirnames):
            if d.lower() == slug.lower():
                hits.append(Path(dirpath) / d)
    if owner:
        hits.sort(key=lambda p: (owner not in p.parts, len(p.parts), str(p)))
    return hits[0] if hits else None


def find_test(competition):
    """The competition's test.parquet (replaced by the hidden test on a scoring rerun)."""
    comp = find_dataset(competition)
    if comp is not None:
        hits = sorted(comp.rglob("test.parquet"))
        if hits:
            return hits[0]
    hits = sorted(INPUT.rglob("test.parquet"))
    if not hits:
        raise FileNotFoundError("no test.parquet under /kaggle/input")
    return hits[0]

In [ ]:
T0 = time.time()
print("python", sys.version.split()[0], "| cpus", os.cpu_count())
try:
    import rdkit  # noqa: F401
except ImportError:
    owner, slug = RDKIT_WHEEL_DATASET.split("/")
    wd = find_dataset(slug, owner)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-index", "--quiet",
                        f"--find-links={wd}", "rdkit==2026.3.3"], capture_output=True, text=True)
    print("rdkit install", r.returncode, r.stderr[-500:])
CODE = find_dataset(CODE_DATASET, OWNER)
BUNDLE = find_dataset(BUNDLE_DATASET, OWNER)
NPDIR = find_dataset(NP_DATASET, OWNER)
print("code", CODE, "| bundle", BUNDLE, "| np", NPDIR)
assert CODE is not None and (CODE / "casmi" / "encoder.py").is_file(), "code dataset missing"
assert BUNDLE is not None and (BUNDLE / "info.json").is_file(), "bundle dataset missing"
NPFP = sorted(NPDIR.rglob("np_fp.npy"))[0]
sys.path.insert(0, str(CODE))
import torch
N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "cuda", torch.version.cuda, "gpus", N_GPU,
      [torch.cuda.get_device_name(i) for i in range(N_GPU)])
subprocess.run(["nvidia-smi"], check=False)
info = json.loads((BUNDLE / "info.json").read_text())
print("bundle:", {k: info.get(k) for k in ("created", "n_structures", "n_spectra", "n_groups",
                                            "holdout_sha256", "git_commit")})
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "bundle_info.json").write_text(json.dumps(info, indent=1))

In [ ]:
parallel = N_GPU >= len(ARMS)
hours = HOURS if parallel else HOURS / max(1, len(ARMS))
env0 = dict(os.environ, PYTHONPATH=str(CODE), CASMI_ROOT=str(WORK / "casmi_run"),
            NUMBA_CACHE_DIR=str(WORK / "numba_cache"), PYTHONUNBUFFERED="1")


def command(arm):
    cmd = [sys.executable, str(CODE / "scripts" / "train_encoder.py"), "--arm", arm,
           "--hours", str(hours), "--bundle", str(BUNDLE), "--np-fp", str(NPFP),
           "--out", str(OUT / arm), "--device", "cuda:0" if N_GPU else "cpu",
           "--eval-max-mol", str(EVAL_MAX_MOL)]
    for k, v in sorted(SETS.items()):
        cmd += ["--set", f"{k}={json.dumps(v)}"]
    return cmd


def launch(i, arm):
    (OUT / arm).mkdir(parents=True, exist_ok=True)
    env = dict(env0, CUDA_VISIBLE_DEVICES=str(i % max(N_GPU, 1)))
    fh = open(OUT / arm / "stdout.log", "w")
    print("launch", arm, "on gpu", env["CUDA_VISIBLE_DEVICES"], "hours", hours, flush=True)
    return subprocess.Popen(command(arm), env=env, stdout=fh, stderr=subprocess.STDOUT), fh


def tail(arm, n=2):
    p = OUT / arm / "stdout.log"
    try:
        lines = [l for l in p.read_text(errors="replace").splitlines() if "INFO" in l or "Error" in l]
    except OSError:
        return []
    return lines[-n:]


def wait(procs):
    last = 0.0
    while any(p.poll() is None for p, _, _ in procs):
        time.sleep(20)
        if time.time() - last > 600:
            last = time.time()
            for p, _, arm in procs:
                print(f"[{(time.time() - T0) / 60:.0f} min] {arm} rc={p.poll()}", *tail(arm), sep="\n  ",
                      flush=True)
    for p, fh, arm in procs:
        fh.close()
        print(arm, "exit code", p.returncode, flush=True)


if parallel:
    procs = []
    for i, arm in enumerate(ARMS):
        p, fh = launch(i, arm)
        procs.append((p, fh, arm))
    wait(procs)
else:
    for i, arm in enumerate(ARMS):
        p, fh = launch(0, arm)
        wait([(p, fh, arm)])

In [ ]:
for arm in ARMS:
    print("=" * 30, arm)
    for l in tail(arm, 6):
        print(l)
    sp = OUT / arm / "summary.json"
    if sp.exists():
        s = json.loads(sp.read_text())
        print(json.dumps({k: s.get(k) for k in ("steps", "train_seconds", "wall_seconds", "n_params")}))
        print(json.dumps(s.get("final", {}), indent=1))
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:9.1f} MB  {p.relative_to(WORK)}")
print(f"total {(time.time() - T0) / 60:.1f} min")